# 2. Baseline: the alignment-based reconstruction of the MPE manuscript on the test families

For every family of the test set the ancestor is reconstructed from D1 and D2 by the pipeline of the MPE manuscript (`reconstruct_multi`: the two sibling alignments, choice of direction, majority vote), run exactly as in `tcbb_exp1_cell`. Recorded per family: Â, e(A, Â), n(Â) − n(A), and for each descendant d_est(Â, Dᵢ) and d_est(A, Dᵢ) under the match-reward and the edit-weight scorings, with d_hist(A, Dᵢ).

Runtime: CPU only. A family takes from under a second (s = 2) to about a minute (s = 20); the 1,000 test families take roughly 3 hours in one session, proportionally less with parallel sessions. No GPU is needed, so this notebook can run while the models train.

In [ ]:
#@title Setup: fetch the code and choose where results are kept
REPO_URL = "https://github.com/AtheerMatroud/TandemRepeats"  #@param {type:"string"}
SAVE_TO_DRIVE = True  #@param {type:"boolean"}

import os, sys, subprocess, zipfile, io
if not os.path.exists("/content/TandemRepeats/AI/tr_ai"):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "/content/TandemRepeats"])
if not os.path.exists("/content/TandemRepeats/AI/tr_ai"):
    from google.colab import files
    print("The AI folder was not found in the repository. Upload AI.zip instead.")
    up = files.upload(); name = next(iter(up))
    os.makedirs("/content/TandemRepeats", exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(up[name])) as z: z.extractall("/content/TandemRepeats")
AI = "/content/TandemRepeats/AI"
sys.path.insert(0, AI)
os.environ["PYTHONPATH"] = AI + ":" + os.environ.get("PYTHONPATH", "")
if SAVE_TO_DRIVE:
    from google.colab import drive; drive.mount("/content/drive")
    ROOT = "/content/drive/MyDrive/tandem_repeats/ai_experiment"
else:
    ROOT = "/content/ai_experiment"
os.makedirs(ROOT, exist_ok=True)
import json, glob, time, numpy as np, pandas as pd
print("code:", AI, "| results:", ROOT, "| CPU cores:", os.cpu_count())


In [ ]:
#@title Parallel settings
# One session already uses every CPU core. To go faster, open this notebook in several
# Colab sessions at once and give each a different PART (0, 1, ..., N_PARTS - 1).
# All sessions write to the same Drive folder; finished shards are skipped, so an
# interrupted session can simply be run again.
PART = 0     #@param {type:"integer"}
N_PARTS = 1  #@param {type:"integer"}


In [ ]:
#@title Run the baseline on the test set (parallel, resumable)
from tr_ai import simdata, evaluate as ev
fams = {f["id"]: f for f in simdata.load(f"{ROOT}/families/shard_*.jsonl.gz")}
splits = json.load(open(f"{ROOT}/splits.json"))
test = sorted((fams[i] for i in splits["test"]), key=lambda f: -len(f["A"]))
print(len(test), "test families")
ev.run_sharded(ev.baseline_family, test, f"{ROOT}/baseline_test", shard_size=10, part=PART, n_parts=N_PARTS)


In [ ]:
#@title Summary (merges every finished shard)
base = pd.DataFrame(ev.read_shards(f"{ROOT}/baseline_test"))
print(len(base), "of", len(test), "test families done")
print("Accuracy of the reconstructed ancestor, alignment-based pipeline, test set"); display(ev.accuracy_table(base))
print("For reference: the k = 2 families of the MPE manuscript (recorded results)")
display(ev.accuracy_table(ev.mpe_frame(f"{AI}/data/mpe_k2_results.csv")))
base.drop(columns=["Ahat"]).to_csv(f"{ROOT}/baseline_test.csv", index=False)
